In [1]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import cv2

In [4]:
def load_timestamps(text_path):
    with open(text_path, 'r') as f:
        lines = f.readlines()

    timestamps = []
    for line in lines:
        ts = line.strip()
        if '.' in ts:
            base, fraction = ts.split('.')
            # trim or pad to 6 digits
            fraction = (fraction + "000000")[:6]
            clean_ts = f"{base}.{fraction}"
        else:
            clean_ts = ts + ".000000"
        
        timestamps.append(datetime.strptime(clean_ts, "%Y-%m-%d %H:%M:%S.%f"))
    
    return timestamps

In [5]:
# Load KITTI raw point clouds
def load_kitti_point_cloud(bin_path):
    point_cloud = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)
    return point_cloud[:, :3]

In [6]:
# Visualize using Open3D
def visualize_point_cloud(points):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # simple vis
    o3d.visualization.draw_geometries([pcd])

In [7]:
# Rotation matrix around X-axis
def rotate_x(points, angle_degrees):
    theta = np.radians(angle_degrees)
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(theta), -np.sin(theta)],
        [0, np.sin(theta), np.cos(theta)]
    ])
    return points @ Rx.T

In [8]:
def rotate_y(points, angle_degrees):
    theta = np.radians(angle_degrees)
    Ry = np.array([
        [np.cos(theta), np.sin(theta)],
        [0, 1, 0],
        [-np.sin(theta), 0, np.cos(theta)]
    ])
    return points @ Ry.T

In [9]:
def rotate_z(points, angle_degrees):
    theta = np.radians(angle_degrees)
    Rz = np.array([
        [np.cos(theta), -np.sin(theta), 0],
        [np.sin(theta), np.cos(theta), 0],
        [0, 0, 1]
    ])
    return points @ Rz.T

In [10]:
path = "/Users/oktavianu/lidar-3d-object-detection/data/velodyne_points/data/0000000000.bin"
points = load_kitti_point_cloud(path)

In [11]:
rotated = rotate_z(points, 45)
visualize_point_cloud(rotated)

In [12]:
import time 
import os

In [13]:
lidar_dir = "/Users/oktavianu/lidar-3d-object-detection/data/velodyne_points/data"

In [14]:
timestamps = load_timestamps("/Users/oktavianu/lidar-3d-object-detection/data/velodyne_points/timestamps.txt")
for ts in timestamps:
    print(ts)

2011-09-26 14:18:15.059587
2011-09-26 14:18:15.162862
2011-09-26 14:18:15.266167
2011-09-26 14:18:15.370205
2011-09-26 14:18:15.473451
2011-09-26 14:18:15.575953
2011-09-26 14:18:15.679255
2011-09-26 14:18:15.782541
2011-09-26 14:18:15.885819
2011-09-26 14:18:15.989126
2011-09-26 14:18:16.092428
2011-09-26 14:18:16.196449
2011-09-26 14:18:16.299775
2011-09-26 14:18:16.402369
2011-09-26 14:18:16.505663
2011-09-26 14:18:16.608997
2011-09-26 14:18:16.712328
2011-09-26 14:18:16.815637
2011-09-26 14:18:16.918933
2011-09-26 14:18:17.022222
2011-09-26 14:18:17.125519
2011-09-26 14:18:17.228817
2011-09-26 14:18:17.332123
2011-09-26 14:18:17.435431
2011-09-26 14:18:17.538736
2011-09-26 14:18:17.642016
2011-09-26 14:18:17.745293
2011-09-26 14:18:17.848597
2011-09-26 14:18:17.951899
2011-09-26 14:18:18.055233
2011-09-26 14:18:18.158548
2011-09-26 14:18:18.263245
2011-09-26 14:18:18.366570
2011-09-26 14:18:18.468464
2011-09-26 14:18:18.571752
2011-09-26 14:18:18.675029
2011-09-26 14:18:18.778287
2

In [15]:
files = sorted([f for f in os.listdir(lidar_dir) if f.endswith('.bin')])

In [16]:
# animate through first N frames
N = 50

vis = o3d.visualization.Visualizer()
vis.create_window(window_name="KITTI LIDAR sequence")

geom_added = False
pcd = o3d.geometry.PointCloud()

for i in range(N):
    frame_path = os.path.join(lidar_dir, files[i])
    points = load_kitti_point_cloud(frame_path)
    # Optional: downsample to speed up
    pcd.points = o3d.utility.Vector3dVector(points)
    if not geom_added:
        vis.add_geometry(pcd)
        geom_added = True
    else:
        vis.update_geometry(pcd)

    vis.poll_events()
    vis.update_renderer()
    time.sleep(0.1)

vis.destroy_window()

KeyboardInterrupt: 

In [17]:
def color_by_height(points):
    z_values = points[:, 2]
    z_min, z_max = np.min(z_values), np.max(z_values)

    # Normalize z values to range [0, 1]
    normalized = (z_values - z_min) / (z_max - z_min + 1e-6)

    # map to RGB colors (colormap style)
    colors = plt.get_cmap('viridis')(normalized)[:, :3]
    return colors

In [18]:
def timestamps_text(text, pisition=[0, 0, 0], size=0.5):
    text3d = o3d.geometry.Text3D(
        text = str(text),
        font_size=30,
        pos=position,
        direction=[0.0, 0.0, 1.0],
        up=[0.0, -1.0, 0.0],
        font="Monospace",
        density=1,
        width=1.0,
        height=1.0,
    )
    return text3d

In [26]:
# animate through first N frames
N = 100

vis = o3d.visualization.Visualizer()
vis.create_window(window_name="KITTI LIDAR sequence")

geom_added = False
pcd = o3d.geometry.PointCloud()

for i in range(N):
    frame_path = os.path.join(lidar_dir, files[i])

    points = load_kitti_point_cloud(frame_path)
    colors = color_by_height(points)
    # Optional: downsample to speed up
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)

    if not geom_added:
        vis.add_geometry(pcd)
        geom_added = True
    else:
        vis.update_geometry(pcd)

    vis.poll_events()
    vis.update_renderer()
    time.sleep(0.1)

vis.destroy_window()

KeyboardInterrupt: 

In [20]:
import cv2
import numpy as np
import open3d as o3d

def render_point_cloud_with_timestamp(points, timestamp_str, window_size=(720, 720)):
    # Convert points to Open3D format
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.paint_uniform_color([0.2, 0.7, 0.8])  # cyan-ish

    # Setup visualizer
    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False, width=window_size[0], height=window_size[1])
    vis.add_geometry(pcd)
    vis.poll_events()
    vis.update_renderer()

    # Capture image from Open3D window
    image = vis.capture_screen_float_buffer(False)
    vis.destroy_window()

    # Convert float buffer to uint8 image
    image = (255 * np.asarray(image)).astype(np.uint8)

    # Add timestamp using OpenCV
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(
        image,
        f"{timestamp_str}",
        (10, 30),
        font,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    # Convert RGB to BGR for OpenCV
    image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image_bgr


In [22]:
points = load_kitti_point_cloud("/Users/oktavianu/lidar-3d-object-detection/data/velodyne_points/data/0000000000.bin")
timestamp = timestamps[0].strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

frame = render_point_cloud_with_timestamp(points, timestamp)

# Show it
cv2.imshow("LIDAR Frame", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()
